# ローカル LLM 回答役の比較（Issue #1・Colab 用）

正解文書を文脈として渡し、日本語で根拠付き回答を書けるかを比べる（検索は含めない）。
データは架空の社内規程（evaluation/corpus）。ランタイムは GPU（T4 で可）。

- 指標: 期待キーワード全一致率 / 部分一致率 / 平均応答秒 / VRAM ピーク（4bit 読み込み時）
- VRAM ピークが 6GB 未満でも、Laya との同居分（約 1GB）を引いて判断する
- T4 と RTX 3050 は速度が違うので、応答秒は相対比較にだけ使う

In [ ]:
!pip -q install -U transformers accelerate bitsandbytes
!git clone -q --depth 1 -b redesign/v2 https://github.com/Maaa2005/local-rag-agent.git repo

In [ ]:
MODELS = [
    "google/gemma-4-E2B-it",
    "sbintuitions/sarashina2.2-3b-instruct-v0.1",
    "Qwen/Qwen3.5-2B",
    "Qwen/Qwen3.5-4B",
]
QUANT_4BIT = True   # AWQ-INT4 相当の容量を見るため 4bit で読む
MAX_NEW_TOKENS = 384

In [ ]:
import json, time, gc, pathlib, torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

root = pathlib.Path("repo/evaluation")
qs = [json.loads(l) for l in open(root / "dataset.jsonl", encoding="utf-8")]
qs = [q for q in qs if not q.get("should_refuse")]
def doc(src):  # /watched/general/x.md -> corpus/general/x.md
    return (root / "corpus" / src.removeprefix("/watched/")).read_text(encoding="utf-8")
SYSTEM = "あなたは社内文書アシスタントです。与えられた文書だけを根拠に日本語で簡潔に答え、根拠の文書名を示してください。文書に無いことは「文書に記載がありません」と答えてください。"
print(len(qs), "questions")

In [ ]:
def run(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    kw = dict(device_map="cuda", torch_dtype=torch.float16)
    if QUANT_4BIT:
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16, bnb_4bit_quant_type="nf4")
    torch.cuda.reset_peak_memory_stats()
    model = AutoModelForCausalLM.from_pretrained(model_id, **kw)
    rows = []
    for q in qs:
        ctx = "\n\n".join(f"### {s.split('/')[-1]}\n{doc(s)}" for s in q["expected_sources"])
        msgs = [{"role": "system", "content": SYSTEM},
                {"role": "user", "content": f"{ctx}\n\n質問: {q['question']}"}]
        ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", enable_thinking=False).to("cuda")
        t = time.time()
        out = model.generate(ids, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
        ans = tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True)
        hit = [k in ans for k in q["expected_answer_keywords"]]
        rows.append(dict(id=q["id"], sec=time.time() - t, all=all(hit), part=sum(hit) / len(hit), ans=ans))
    peak = torch.cuda.max_memory_allocated() / 2**30
    del model; gc.collect(); torch.cuda.empty_cache()
    return rows, peak

results = {}
for m in MODELS:
    try:
        results[m] = run(m)
    except Exception as e:
        print(m, "FAILED:", e)

In [ ]:
print(f"{'model':45} {'全一致':>6} {'部分':>6} {'秒':>6} {'VRAM GB':>8}")
for m, (rows, peak) in results.items():
    n = len(rows)
    print(f"{m:45} {sum(r['all'] for r in rows)/n:6.2f} {sum(r['part'] for r in rows)/n:6.2f} {sum(r['sec'] for r in rows)/n:6.1f} {peak:8.2f}")
json.dump({m: r for m, (r, _) in results.items()}, open("answers.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("answers.json に全回答を保存（目視で根拠の示し方も確認する）")